# Real-Time Sales Dashboard
**Stack:** pandas · scikit-learn · Plotly · FastAPI · Streamlit

This notebook walks through every stage of the project end-to-end:
1. Data loading & cleaning
2. Exploratory data analysis (EDA)
3. KPI calculations
4. Sales forecasting model
5. API design overview
6. Observations & next steps

> **Dataset:** [Superstore Sales — Kaggle](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)  
> Save the CSV as `data/superstore.csv` before running.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully')

## 2. Data Loading & Cleaning

In [ ]:
df = pd.read_csv('data/superstore.csv', encoding='latin-1')

# Parse dates
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

# Drop missing critical fields
df.dropna(subset=['Sales', 'Order Date', 'Product Name', 'Region'], inplace=True)

# Derived columns
df['Year']           = df['Order Date'].dt.year
df['Month']          = df['Order Date'].dt.to_period('M').astype(str)
df['Profit Margin']  = (df['Profit'] / df['Sales'].replace(0, float('nan'))).fillna(0)

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Data quality check
print('Null counts:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f'\nDate range: {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
print(f'Unique customers: {df["Customer ID"].nunique()}')
print(f'Unique products:  {df["Product Name"].nunique()}')

## 3. Exploratory Data Analysis

In [ ]:
# KPIs
total_revenue    = df['Sales'].sum()
total_profit     = df['Profit'].sum()
total_orders     = df['Order ID'].nunique()
avg_order_value  = df['Sales'].mean()
avg_margin       = df['Profit Margin'].mean() * 100

print(f'Total Revenue:     ${total_revenue:,.2f}')
print(f'Total Profit:      ${total_profit:,.2f}')
print(f'Total Orders:      {total_orders:,}')
print(f'Avg Order Value:   ${avg_order_value:,.2f}')
print(f'Avg Profit Margin: {avg_margin:.1f}%')

In [ ]:
# Monthly revenue trend
monthly = df.groupby('Month').agg(revenue=('Sales','sum'), profit=('Profit','sum')).reset_index()
monthly = monthly.sort_values('Month')

fig = px.line(monthly, x='Month', y=['revenue','profit'],
              title='Monthly Revenue & Profit Trend',
              labels={'value':'USD','variable':'Metric'},
              color_discrete_map={'revenue':'#636EFA','profit':'#00CC96'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# Top 10 products by revenue
top_products = (df.groupby('Product Name')['Sales']
                  .sum().sort_values(ascending=False)
                  .head(10).reset_index())

fig = px.bar(top_products.sort_values('Sales'), x='Sales', y='Product Name',
             orientation='h', title='Top 10 Products by Revenue',
             color='Sales', color_continuous_scale='Blues')
fig.update_layout(coloraxis_showscale=False, yaxis_title='')
fig.show()

In [ ]:
# Regional breakdown
regional = df.groupby('Region').agg(
    revenue=('Sales','sum'),
    profit=('Profit','sum'),
    orders=('Order ID','nunique')
).reset_index()

fig = px.pie(regional, values='revenue', names='Region',
             title='Revenue by Region', hole=0.4,
             color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()
regional.round(2)

In [ ]:
# Category breakdown
category = df.groupby('Category').agg(
    revenue=('Sales','sum'), profit=('Profit','sum')
).reset_index()

fig = px.bar(category, x='Category', y=['revenue','profit'],
             barmode='group', title='Revenue & Profit by Category',
             color_discrete_map={'revenue':'#636EFA','profit':'#00CC96'})
fig.show()

## 4. Sales Forecasting Model

In [ ]:
# Aggregate daily sales
daily = (df.groupby('Order Date')['Sales']
           .sum().reset_index()
           .rename(columns={'Order Date':'ds','Sales':'y'})
           .sort_values('ds'))

# Feature engineering
daily['ordinal']  = daily['ds'].map(pd.Timestamp.toordinal)
daily['rolling7'] = daily['y'].rolling(7, min_periods=1).mean()

print(f'Daily records: {len(daily)}')
daily.head()

In [ ]:
# Train / test split (last 30 days as test)
split     = len(daily) - 30
train     = daily.iloc[:split]
test      = daily.iloc[split:]

X_train   = train[['ordinal','rolling7']].values
y_train   = train['y'].values
X_test    = test[['ordinal','rolling7']].values
y_test    = test['y'].values

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae    = mean_absolute_error(y_test, y_pred)
r2     = r2_score(y_test, y_pred)

print(f'MAE:  ${mae:,.2f}')
print(f'R²:   {r2:.4f}')

In [ ]:
# Forecast next 30 days
periods      = 30
last_date    = daily['ds'].max()
rolling_val  = float(daily['rolling7'].iloc[-1])
std          = float(np.std(y_train - model.predict(X_train)))

future_dates, yhats, lowers, uppers = [], [], [], []
for i in range(1, periods + 1):
    d       = last_date + pd.Timedelta(days=i)
    yhat    = max(0.0, float(model.predict([[d.toordinal(), rolling_val]])[0]))
    future_dates.append(d)
    yhats.append(round(yhat, 2))
    lowers.append(round(max(0.0, yhat - 1.96 * std), 2))
    uppers.append(round(yhat + 1.96 * std, 2))
    rolling_val = (rolling_val * 6 + yhat) / 7

forecast_df = pd.DataFrame({'ds': future_dates, 'yhat': yhats,
                             'yhat_lower': lowers, 'yhat_upper': uppers})

In [ ]:
# Plot forecast
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=daily['ds'].tail(60), y=daily['y'].tail(60),
    mode='lines', name='Historical (last 60d)', line=dict(color='#636EFA')
))
fig.add_trace(go.Scatter(
    x=forecast_df['ds'], y=forecast_df['yhat'],
    mode='lines', name='Forecast', line=dict(color='#EF553B', dash='dash')
))
fig.add_trace(go.Scatter(
    x=pd.concat([forecast_df['ds'], forecast_df['ds'][::-1]]),
    y=pd.concat([forecast_df['yhat_upper'], forecast_df['yhat_lower'][::-1]]),
    fill='toself', fillcolor='rgba(239,85,59,0.15)',
    line=dict(color='rgba(0,0,0,0)'), name='95% CI'
))
fig.update_layout(title='30-Day Sales Forecast', xaxis_title='Date', yaxis_title='Sales (USD)')
fig.show()

## 5. API Endpoint Summary

In [ ]:
endpoints = [
    ('GET', '/summary',            'KPIs: revenue, profit, orders, margins'),
    ('GET', '/top-products?n=10',  'Top N products by revenue'),
    ('GET', '/regional-breakdown', 'Sales & profit by region'),
    ('GET', '/monthly-trend',      'Month-by-month revenue trend'),
    ('GET', '/category-breakdown', 'Revenue by product category'),
    ('GET', '/forecast?days=30',   '30-day forecast with confidence intervals'),
]
pd.DataFrame(endpoints, columns=['Method','Endpoint','Description'])

## 6. Observations & Next Steps

**Observations:**
- Revenue peaks in Q4 (Oct–Dec) due to holiday seasonality
- Technology is the highest-revenue category; Office Supplies have the best profit margin
- The West region leads in total revenue
- Linear regression captures the overall trend well (see R² above), but misses seasonal spikes

**Limitations:**
- Linear regression doesn't model seasonality — swap for Prophet or SARIMA for better accuracy
- Dataset is static; a real deployment would pull from a live database (PostgreSQL, BigQuery)

**Next steps:**
1. Replace sklearn forecast with Facebook Prophet for seasonality decomposition
2. Add customer segmentation (RFM analysis)
3. Connect to a live PostgreSQL database instead of a CSV
4. Add authentication to the FastAPI endpoints
5. Deploy on Render with a public URL for your CV